In [1]:
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa
from scipy.stats import wilcoxon
import numpy as np

%load_ext autoreload
%autoreload 2

In [2]:
data = pd.read_csv("RQ3_anonymized.csv", index_col=0)
batch1, batch2 = data[data.iloc[:,0]==1].iloc[:, 1:], data[data.iloc[:,0]==2].iloc[:, 1:]

In [3]:
print(f"Num batch 1: {len(batch1)}, Num batch 2: {len(batch2)}")

Num batch 1: 11, Num batch 2: 9


In [4]:
mapping = {'Yes': 1., 'No': 0., 'Unsure': 0.5}

all_mrm_unsure, all_mimicry_unsure, all_hynea_unsure = 0, 0, 0
all_mrm_lp, all_mimicry_lp, all_hynea_lp = [], [], []
all_mrm_real, all_mimicry_real, all_hynea_real = [], [], []

interr = []
for i, b in enumerate([batch1, batch2]):
    realism = b.iloc[:, ::2].dropna(axis=1, how='all')

    content = b.iloc[:, 1::2]
    content = content.dropna(axis=1, how='all')
    content_agreement = content.replace(mapping)

    ccm = content_agreement.apply(lambda r: r.value_counts().reindex([0.,0.5,1.], fill_value=0), axis=1)
    interr.append(fleiss_kappa(ccm.values, method="rand"))


    mrm_res, mimicry_res, hynea_res = [], [], []
    lists = [mrm_res, mimicry_res, hynea_res]
    for c in content_agreement.columns[::-1]:
        shortest = lists.index(min(lists, key=len))
        col = content_agreement[c]
        c_len = len(col)
        res = (col).tolist()

        n_unsure = (col == 0.5).sum()
        if shortest == 0:
            if len(mrm_res) < 2*c_len:
                mrm_res.extend(res)
                all_mrm_unsure += n_unsure
        elif shortest == 1:
            mimicry_res.extend(res)
            all_mimicry_unsure += n_unsure
        else:
            hynea_res.extend(res)
            all_hynea_unsure += n_unsure

    all_mrm_lp.extend(mrm_res)
    all_mimicry_lp.extend(mimicry_res)
    all_hynea_lp.extend(hynea_res)

    mrm_res, mimicry_res, hynea_res = [], [], []
    lists = [mrm_res, mimicry_res, hynea_res]

    for c in realism.columns[::-1]:
        shortest = lists.index(min(lists, key=len))
        res = ((realism[c] - 1) / 4).tolist()

        if shortest == 0:
            if len(mrm_res) < 2:
                mrm_res.extend(res)
        elif shortest == 1:
            mimicry_res.extend(res)
        else:
            hynea_res.extend(res)

    all_mrm_real.extend(mrm_res)
    all_mimicry_real.extend(mimicry_res)
    all_hynea_real.extend(hynea_res)

print("\n== Aggregated Across Batches ==")
print(
    f"GIFTBench label preservation: {np.mean(all_mrm_lp):.3f} pm {np.std(all_mrm_lp):.3f}\n"
    f"Mimicry label preservation: {np.mean(all_mimicry_lp):.3f} pm {np.std(all_mimicry_lp):.3f}\n"
    f"HyNeA label preservation: {np.mean(all_hynea_lp):.3f} pm {np.std(all_hynea_lp):.3f}\n"
)

print(
    f"GIFTBench realism: {np.mean(all_mrm_real):.3f} pm {np.std(all_mrm_real):.3f}\n"
    f"Mimicry realism: {np.mean(all_mimicry_real):.3f} pm {np.std(all_mimicry_real):.3f}\n"
    f"HyNeA realism: {np.mean(all_hynea_real):.3f} pm {np.std(all_hynea_real):.3f}"
)

print("\nPercentage of 'Unsure' responses:")
print(f"GIFTBench: {all_mrm_unsure / len(all_mrm_lp):.3f}")
print(f"Mimicry: {all_mimicry_unsure / len(all_mimicry_lp):.3f}")
print(f"HyNeA: {all_hynea_unsure /len(all_hynea_lp):.3f}")

print(f"\nInterrater agreement: {np.mean(interr):.3f} pm {np.std(interr):.3f}")


== Aggregated Across Batches ==
GIFTBench label preservation: 0.463 pm 0.479
Mimicry label preservation: 0.287 pm 0.431
HyNeA label preservation: 0.787 pm 0.401

GIFTBench realism: 0.438 pm 0.361
Mimicry realism: 0.388 pm 0.340
HyNeA realism: 0.750 pm 0.237

Percentage of 'Unsure' responses:
GIFTBench: 0.075
Mimicry: 0.075
HyNeA: 0.025

Interrater agreement: 0.336 pm 0.053


/tmp/ipykernel_3041224/1703907123.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  content_agreement = content.replace(mapping)
/tmp/ipykernel_3041224/1703907123.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  content_agreement = content.replace(mapping)


In [5]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2)/(nx+ny-2))
    return (np.mean(x) - np.mean(y)) / pooled_std

for metric, hy, other1, other2 in [('Label preservation', all_hynea_lp, all_mrm_lp, all_mimicry_lp),
                                   ('Realism', all_hynea_real, all_mrm_real, all_mimicry_real)]:
    print(f"\n{metric}:")
    print(f"HyNeA vs MRM: p={wilcoxon(hy, other1)[1]:.1e}, Cohen's d={cohens_d(hy, other1):.3f}")
    print(f"HyNeA vs Mimicry: p={wilcoxon(hy, other2)[1]:.1e}, Cohen's d={cohens_d(hy, other2):.3f}")


Label preservation:
HyNeA vs MRM: p=8.8e-03, Cohen's d=0.726
HyNeA vs Mimicry: p=9.6e-06, Cohen's d=1.185

Realism:
HyNeA vs MRM: p=6.1e-03, Cohen's d=0.997
HyNeA vs Mimicry: p=2.7e-03, Cohen's d=1.206
